In [ ]:
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf

In [ ]:
tfds.disable_progress_bar()

In [ ]:
tf.__version__

'2.19.0'

In [ ]:
device=tf.config.experimental.list_physical_devices('GPU')
device

[]

In [ ]:
physical_devices=tf.config.list_physical_devices('GPU')
try:
  tf.config.experimental.set_memory_growth(device[0],True)
  print("Success")
except:
  print("exception occured")
  pass

exception occured


In [ ]:
dataset,info=tfds.load('imdb_reviews',data_dir='.',with_info=True,as_supervised=True)
info

Dataset imdb_reviews downloaded and prepared to imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples=25000, num_shards=

In [ ]:
dataset

{Split('train'): <_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 Split('test'): <_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 Split('unsupervised'): <_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>}

In [ ]:
train_dataset,test_dataset=dataset['train'],dataset['test']
type(train_dataset)

tensorflow.python.data.ops.prefetch_op._PrefetchDataset

In [ ]:
len(train_dataset)

25000

In [ ]:
len(test_dataset)

25000

In [ ]:
for sample in train_dataset:
  print(sample[0].numpy())
  print(sample[1].numpy())
  break

b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it."
0


In [ ]:
BUFFER_SIZE=10000
BATCH_SIZE=64

In [ ]:
train_dataset=train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset=test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
for example,label in train_dataset.take(1):
  print('texts: ',example.numpy()[:3])
  print()
  print('labels: ',label.numpy()[:3])

texts:  [b'I was expecting a little something from "K-911", I mean it did look like a cute movie that I could get into. I always did love the dog comedy movies. But it looked like it was supposed to be Jame\'s movie, not Jerry Lee\'s. The plot was pretty lame and the two love interests really didn\'t have chemistry to begin with. Not to mention that James seemed to have a total sexist view in the movie despite the fact the writer wasn\'t going in that direction. James just really ticked me off for more than half the film. The dogs were the true stars and that\'s pretty sad that they out shined the actors.<br /><br />So, I\'m glad it\'s not just me on IMDb who agrees that this was a pretty stupid movie. But hopefully, James will realize it was his brother Jim who was the talented one, no offense, but not everyone can be their star sibling. Don\'t you wish Ashlee Simpson would take that same advice? :D <br /><br />3/10'
 b"I like the cast pretty much however the story sort of unfolds rat

In [ ]:
from tensorflow import keras
e=tf.keras.layers.TextVectorization()
e.adapt([
    "i love biryani and julabjam",
    "i love long bike ride",
    "i love spending and playing with my sister"
])

In [ ]:
e.get_vocabulary()

['',
 '[UNK]',
 np.str_('love'),
 np.str_('i'),
 np.str_('and'),
 np.str_('with'),
 np.str_('spending'),
 np.str_('sister'),
 np.str_('ride'),
 np.str_('playing'),
 np.str_('my'),
 np.str_('long'),
 np.str_('julabjam'),
 np.str_('biryani'),
 np.str_('bike')]

In [ ]:
e(['I love pizza']).numpy()

array([[3, 2, 1]])

In [ ]:
VOCAB_SIZE=1000
encoder=tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE)
encoder.adapt(train_dataset.map(lambda text,label:text))

In [ ]:
vocab=np.array(encoder.get_vocabulary())
vocab[:25]

array(['', '[UNK]', 'the', 'and', 'a', 'of', 'to', 'is', 'in', 'it', 'i',
       'this', 'that', 'br', 'was', 'as', 'for', 'with', 'movie', 'but',
       'film', 'on', 'not', 'you', 'are'], dtype='<U14')

In [ ]:
example[:2]

<tf.Tensor: shape=(2,), dtype=string, numpy=
array([b'I was expecting a little something from "K-911", I mean it did look like a cute movie that I could get into. I always did love the dog comedy movies. But it looked like it was supposed to be Jame\'s movie, not Jerry Lee\'s. The plot was pretty lame and the two love interests really didn\'t have chemistry to begin with. Not to mention that James seemed to have a total sexist view in the movie despite the fact the writer wasn\'t going in that direction. James just really ticked me off for more than half the film. The dogs were the true stars and that\'s pretty sad that they out shined the actors.<br /><br />So, I\'m glad it\'s not just me on IMDb who agrees that this was a pretty stupid movie. But hopefully, James will realize it was his brother Jim who was the talented one, no offense, but not everyone can be their star sibling. Don\'t you wish Ashlee Simpson would take that same advice? :D <br /><br />3/10',
       b"I like the cast

In [ ]:
encoded_example=encoder(example)[:3].numpy()
encoded_example

array([[ 10,  14, 982, ...,   0,   0,   0],
       [ 10,  39,   2, ...,   0,   0,   0],
       [ 10,  41, 208, ...,   0,   0,   0]])

In [ ]:
for n in range(3):
  print("Original:",example[n].numpy())
  print("Round-trip: ".join(vocab[encoded_example[n]]))
  print()

Original: b'I was expecting a little something from "K-911", I mean it did look like a cute movie that I could get into. I always did love the dog comedy movies. But it looked like it was supposed to be Jame\'s movie, not Jerry Lee\'s. The plot was pretty lame and the two love interests really didn\'t have chemistry to begin with. Not to mention that James seemed to have a total sexist view in the movie despite the fact the writer wasn\'t going in that direction. James just really ticked me off for more than half the film. The dogs were the true stars and that\'s pretty sad that they out shined the actors.<br /><br />So, I\'m glad it\'s not just me on IMDb who agrees that this was a pretty stupid movie. But hopefully, James will realize it was his brother Jim who was the talented one, no offense, but not everyone can be their star sibling. Don\'t you wish Ashlee Simpson would take that same advice? :D <br /><br />3/10'
iRound-trip: wasRound-trip: expectingRound-trip: aRound-trip: littl

In [ ]:
model=tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=64,
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(1)

])

In [ ]:
sample_text=('The movie was cool.the animation and the graphics'
'were out of this world.I would recommond this movie.')
sample_text=('Awesome movie,I love it so much')
predictions=model.predict(tf.constant([sample_text]))
print(predictions[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
[0.00362484]


In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=['accuracy']
)

In [ ]:
model.fit(train_dataset,epochs=10,
          validation_data=test_dataset,
          validation_steps=30)

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 703s 2s/step - accuracy: 0.5948 - loss: 0.6197 - val_accuracy: 0.7625 - val_loss: 0.5785
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 743s 2s/step - accuracy: 0.7896 - loss: 0.4325 - val_accuracy: 0.8255 - val_loss: 0.3871
Epoch 3/10
  5/391 ━━━━━━━━━━━━━━━━━━━━ 10:23 2s/step - accuracy: 0.8861 - loss: 0.3120

KeyboardInterrupt: 